1. catalog를 보고 필요한 pdf가 뭔지 찾는 tool
2.받은 question을 기반으로 특정 pdf에서 해당구간의 chunk를 찾는 tool. 

pdf 폴더 안에다가 pdf 모아놓고 llm 한테 q 주면 적합한 답을 찾게 한다

q와 관련된 pdf를 고르는 tool
-지금 현재 낸가 무슨 pdf가 있는지 알아야 함 -> pdf 파일이 뭐가 있는지 알려주는 tool
- 각 pdf가 어떤 내용인지 적당히 알아야함 - pdf 요약하는 tool + 요약본을 저장하는 tool (얘는 이미 되어있어)
-pdf를 선정해준다 -> 요약을 읽고 pdf 하나 골라주는 tool
-pdf 내에 q와 관련된 내용을 찾는 tool -> rag

In [ ]:
import json
import os
import re
import sys
from datetime import datetime
from pathlib import Path

import pytz
import yfinance as yf
from dotenv import load_dotenv
from openai import OpenAI

BASE = Path('.').resolve()
sys.path.insert(0, str(BASE))

load_dotenv("../../.env")
api_key = os.getenv("OPENAI_KEY")
client = OpenAI(api_key=api_key)

SAMPLES = BASE / 'samples' / 'pdf_samples'
print('✅ client 준비')

In [ ]:
#RAG용 함수 정의 

def split_text(text, chunk_size=280, overlap=60):
    text = re.sub(r'\s+', ' ', text.strip())
    if len(text) <= chunk_size:
        return [text]
    chunks, start = [], 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start = max(0, end - overlap)
    return chunks

def tokenize(text):
    return {t.lower() for t in re.findall(r'[가-힣a-zA-Z0-9]+', text) if len(t) > 1}

def search(query, top_k=3):
    q = tokenize(query)
    scored = []
    for item in INDEX:
        score = len(q & tokenize(item['text']))
        if score > 0:
            scored.append((score, item))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [{'score': s, **it} for s, it in scored[:top_k]]

